# 🎌 ロール別パフォーマンス分析

v18 11役職の使用状況・レスポンスタイム・モデル別統計を分析します。

**データソース:** `messages` テーブル（agent_role, execution_time, model）

In [ ]:
import utils
from utils import qdf, scalar, ROLE_JA, ROLE_MODEL, PALETTE, ROLE_COLOR
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

## 1. ロール別 使用回数・レスポンスタイム

In [ ]:
df = qdf("""
    SELECT
        agent_role,
        COUNT(*)                                                      AS cnt,
        ROUND(AVG(execution_time)::numeric, 1)                        AS avg_ms,
        ROUND(PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY execution_time)::numeric, 1) AS median_ms,
        ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY execution_time)::numeric, 1) AS p95_ms,
        ROUND(MIN(execution_time)::numeric, 1)                        AS min_ms,
        ROUND(MAX(execution_time)::numeric, 1)                        AS max_ms
    FROM messages
    WHERE role = 'assistant'
      AND agent_role IS NOT NULL AND agent_role != ''
      AND execution_time > 0
    GROUP BY agent_role
    ORDER BY cnt DESC
""")

if utils.check_data(df, 'ロール別データ'):
    df['役職'] = df['agent_role'].map(lambda r: f"{ROLE_JA.get(r, r)} ({r})")
    df['モデル'] = df['agent_role'].map(ROLE_MODEL)
    print(f'集計ロール数: {len(df)} / 11')
    display(df[['役職','モデル','cnt','avg_ms','median_ms','p95_ms']].rename(
        columns={'cnt':'件数','avg_ms':'平均ms','median_ms':'中央値ms','p95_ms':'P95ms'}
    ).set_index('役職'))

## 2. ロール別 使用回数 vs 平均レスポンスタイム

In [ ]:
if utils.check_data(df, 'ロール別データ') and len(df) > 0:
    colors = [ROLE_COLOR.get(r, '#AAAAAA') for r in df['agent_role']]
    labels = [ROLE_JA.get(r, r) for r in df['agent_role']]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 使用回数
    axes[0].barh(labels, df['cnt'], color=colors, edgecolor='white')
    axes[0].set_title('ロール別 応答件数')
    axes[0].set_xlabel('件数')
    for bar in axes[0].patches:
        axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                     f'{bar.get_width():.0f}', va='center', fontsize=9)

    # 平均レスポンスタイム（存在するもの）
    rt_df = df[df['avg_ms'].notna()].copy()
    rt_labels = [ROLE_JA.get(r, r) for r in rt_df['agent_role']]
    rt_colors = [ROLE_COLOR.get(r, '#AAAAAA') for r in rt_df['agent_role']]
    axes[1].barh(rt_labels, rt_df['avg_ms'], color=rt_colors, edgecolor='white', label='平均')
    axes[1].barh(rt_labels, rt_df['p95_ms'] - rt_df['avg_ms'],
                 left=rt_df['avg_ms'], color=[c+'88' for c in rt_colors], edgecolor='white', label='P95')
    axes[1].set_title('ロール別 レスポンスタイム')
    axes[1].set_xlabel('ミリ秒 (ms)')
    axes[1].legend()

    plt.suptitle('v18 ロールパフォーマンス', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. モデル別 使用状況

In [ ]:
model_df = qdf("""
    SELECT
        COALESCE(NULLIF(model,''), '(不明)') AS model,
        COUNT(*) AS cnt,
        ROUND(AVG(execution_time)::numeric, 1) AS avg_ms
    FROM messages
    WHERE role = 'assistant'
    GROUP BY COALESCE(NULLIF(model,''), '(不明)')
    ORDER BY cnt DESC
""")

if utils.check_data(model_df, 'モデル別データ'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # 使用件数
    model_df.set_index('model')['cnt'].sort_values().plot(
        kind='barh', ax=axes[0], color=PALETTE[1], edgecolor='white')
    axes[0].set_title('モデル別 応答件数')
    axes[0].set_xlabel('件数')

    # 円グラフ
    axes[1].pie(model_df['cnt'], labels=model_df['model'],
                autopct='%1.1f%%', colors=PALETTE[:len(model_df)], startangle=90)
    axes[1].set_title('モデル別 使用割合')

    plt.tight_layout()
    plt.show()
    display(model_df.rename(columns={'model':'モデル','cnt':'件数','avg_ms':'平均ms'}).set_index('モデル'))

## 4. スレッド別ロール多様性（どのスレッドで何役職使われたか）

In [ ]:
diversity = qdf("""
    SELECT
        t.title,
        COUNT(DISTINCT m.agent_role) FILTER (WHERE m.agent_role != '') AS unique_roles,
        COUNT(m.id) FILTER (WHERE m.role='assistant')                    AS bot_msgs,
        STRING_AGG(DISTINCT m.agent_role, ', ')
            FILTER (WHERE m.agent_role IS NOT NULL AND m.agent_role != '') AS roles
    FROM threads t
    JOIN messages m ON m.thread_id = t.id
    WHERE NOT t.is_archived
    GROUP BY t.id, t.title
    HAVING COUNT(m.id) FILTER (WHERE m.role='assistant') > 0
    ORDER BY unique_roles DESC, bot_msgs DESC
    LIMIT 15
""")

if utils.check_data(diversity, 'スレッドデータ'):
    diversity['title_short'] = diversity['title'].str[:35]
    display(diversity[['title_short','unique_roles','bot_msgs','roles']].rename(
        columns={'title_short':'スレッド','unique_roles':'使用役職数',
                 'bot_msgs':'応答数','roles':'役職一覧'}
    ).set_index('スレッド'))

    fig, ax = plt.subplots(figsize=(8, 4))
    diversity['unique_roles'].value_counts().sort_index().plot(
        kind='bar', ax=ax, color=PALETTE[0], edgecolor='white')
    ax.set_title('スレッドあたりの使用役職数分布')
    ax.set_xlabel('使用役職数')
    ax.set_ylabel('スレッド数')
    plt.tight_layout()
    plt.show()

## 5. 時系列: ロール別利用推移

In [ ]:
timeline = qdf("""
    SELECT DATE(created_at) AS day, agent_role, COUNT(*) AS cnt
    FROM messages
    WHERE role='assistant' AND agent_role IS NOT NULL AND agent_role != ''
    GROUP BY DATE(created_at), agent_role
    ORDER BY day
""")

if utils.check_data(timeline, 'タイムラインデータ') and len(timeline['day'].unique()) > 1:
    timeline['day'] = pd.to_datetime(timeline['day'])
    pivot = timeline.pivot_table(index='day', columns='agent_role', values='cnt', fill_value=0)
    pivot.columns = [ROLE_JA.get(c, c) for c in pivot.columns]

    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot(ax=ax, marker='o', linewidth=1.5)
    ax.set_title('ロール別 日次利用推移')
    ax.set_ylabel('応答数')
    ax.legend(loc='upper left', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('ℹ️  複数日のデータが揃うと時系列グラフが表示されます')